<a href="https://colab.research.google.com/github/manaswinialakunta08/machine-learning/blob/main/ID3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd


In [7]:
eps=np.finfo(float).eps
data=pd.read_csv('/content/ID3csv__2026_07_10_10_52_51 (1).csv')
print(data)

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No


In [8]:
#To calculate entropy
def find_entropy(df):
  target=df.keys()[-1]
  entopy=0
  values=df[target].unique()
  for value in values:
    fraction=(
        df[target].value_counts()[value]/len(df[target])
    )
    entopy+=-fraction*np.log2(fraction)
  return entopy

In [9]:
#To calculate the entropy of attribute

def find_entropy_attribute(df, attribute):

    target = df.keys()[-1]

    target_variables = df[target].unique()

    variables = df[attribute].unique()

    entropy2 = 0

    for variable in variables:

        entropy = 0

        for target_variable in target_variables:

            num = len(
                df[attribute][
                    df[attribute] == variable
                ][
                    df[target] == target_variable
                ]
            )

            den = len(
                df[attribute][
                    df[attribute] == variable
                ]
            )

            fraction = num / (den + eps)

            entropy += -fraction * np.log2(
                fraction + eps
            )

        fraction2 = den / len(df)

        entropy2 += -fraction2 * entropy

    return abs(entropy2)


In [10]:
#Find attribute with max information gain
def bestClassifier(df):

    IG = []

    for key in df.keys()[:-1]:

        information_gain = (
            find_entropy(df)
            - find_entropy_attribute(df, key)
        )

        IG.append(information_gain)

    return df.keys()[:-1][np.argmax(IG)]

In [11]:
#Get subtable
def get_subtable(df, node, value):

    return df[
        df[node] == value
    ].reset_index(drop=True)

In [12]:
#Build ID3 DecisionTree
def ID3split(df, tree=None):

    target = df.keys()[-1]

    # Find best attribute
    node = bestClassifier(df)

    # Get unique values
    attributeValues = np.unique(df[node])

    # Create tree
    if tree is None:

        tree = {}

        tree[node] = {}

    # Process every attribute value
    for value in attributeValues:

        # Create subtable
        subtable = get_subtable(
            df,
            node,
            value
        )

        # Find target values and counts
        targetValues, counts = np.unique(
            subtable[target],
            return_counts=True
        )

        # If pure node
        if len(counts) == 1:

            tree[node][value] = targetValues[0]

        # Otherwise recursively split
        else:

            tree[node][value] = ID3split(
                subtable
            )

    return tree

In [13]:
#to build DecisionTree
decisionTree = ID3split(data)

print(decisionTree)

{'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}
